# ZTE — results audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/audits/zte_results_audit.ipynb)

**Upload this notebook on its own.** It clones the repo, reads your Drive, runs the audits that need real ZuCo,
and hands you one zip to drop into `res/`. It trains nothing and writes nothing into your run directories.

| Part | What it answers | Needs | Roughly |
| --- | --- | --- | --- |
| **A** | How much of sentence identity does *spelling* give away, on your actual 700-sentence gallery? | the corpus only — **no checkpoint** | minutes |
| **B** | Where does each trained encoder sit against that floor, and against sentence length? | a checkpoint per run | ~5 min each |
| **C** | Everything above, zipped and downloaded | — | seconds |

**Part A is the one that matters.** The piece oracle is a property of the corpus, not of any model, so it needs
nothing trained. If spelling alone resolves your gallery, a sub-word alignment level cannot produce an
interpretable retrieval number, and that is worth knowing before an A100 runs for a fortnight.

## 1 · Provision

Installs `uv`, clones the repo and builds the pinned Python 3.14 venv. The kernel you are typing in is Colab's own older interpreter and never imports `zte`.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main && cd zte
fi
git fetch --depth 1 origin main && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

`colab()` is the only route into the package: it runs one `zte-colab` subcommand in the venv and returns the JSON it printed.

In [ ]:
import json
import os
import platform
import subprocess
from typing import Any


def colab(command: str, *args: str) -> dict[str, Any]:
    """Runs one `zte-colab` subcommand in the provisioned venv and returns the JSON object it printed.

    This is the notebook's only route into ZTE. The package runs on 3.14 inside the uv venv; this kernel is
    Colab's own older interpreter, so it renders payloads rather than computing them.
    """
    argv = ['uv', 'run', 'zte-colab', command, *args]
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

    return json.loads(done.stdout)


# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

ENV = colab('env')
os.environ.update(ENV['env'])

try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is not granted to this notebook
    _hf, _ = None, print(f'HF_TOKEN unavailable ({type(exc).__name__}) — Hub downloads will be unauthenticated.')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    print('HF_TOKEN loaded — authenticated HuggingFace Hub downloads enabled.')

print(f'repo   : {ENV["root"]}')
print(f'venv   : Python {ENV["venv"]["python"]} · zte {ENV["venv"]["zte"]}   ← every `!uv run` command')
print(f'kernel : Python {platform.python_version()}   ← this cell; renders payloads, never imports zte')
print(f'env    : {", ".join(ENV["env"])}')

## 3 · Mount Drive

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

## 4 · Find the session

Point `RESUME_DATE` at the session holding the runs you want audited, or leave it `None` for today's. `DATA_DIR` is the shared `ZuCo Dataset` folder and is never date-stamped.

In [ ]:
RESUME_DATE: str | None = None  # e.g. '2026-08-19'
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'

_resume = ('--resume-date', RESUME_DATE) if RESUME_DATE else ()
SESSION = colab('session', '--drive', ZTE_DRIVE, '--write-mode', 'auto', *_resume)
os.environ.update(SESSION['env'])

DATA_DIR: str = SESSION['data_dir']
DRIVE_ANALYSIS: str = SESSION['drive_analysis']
RUN_DATE: str = SESSION['run_date']
AUDIT_DIR: str = f'{DRIVE_ANALYSIS}/results_audit'

print(f'session  : {RUN_DATE}   (Drive mounted: {SESSION["drive_mounted"]})')
print(f'raw data : {DATA_DIR}   (present: {SESSION["data_dir_present"]})')
print(f'writing  : {AUDIT_DIR}')
if not SESSION['data_dir_present']:
    print('\nZuCo is not at that path. Fix DATA_DIR before running Part A.')

## Part A · The piece oracle, with no model involved

Four brain-free signatures, each scored as a retrieval oracle over your own gallery:

| Signature | What it is told, and nothing else |
| --- | --- |
| `words` | the word count — the documented 5.14-bit length confound |
| `total` | the total sub-word piece count, one integer per sentence |
| `multiset` | the piece counts with their order destroyed |
| `profile` | the ordered per-word piece counts |

`information_bits` is $\log_2 n - \frac{1}{n}\sum_i \log_2 m_i$, where $m_i$ is how many gallery sentences share
sentence *i*'s signature. On 700 sentences the ceiling is $\log_2 700 = 9.4512$ bits.

Also watch **`alignment_coverage`**: it is the fraction of ZuCo words that matched their own reference text. Below
about 0.99 the piece counts are partly wrong and the bits are not trustworthy.

In [ ]:
!uv run zte-audit --root "{DATA_DIR}" --piece-oracle --out "{AUDIT_DIR}/confound_audit.md"

In [ ]:
import pathlib

AUDIT = json.loads(pathlib.Path(f'{AUDIT_DIR}/confound_audit.json').read_text())
PIECE = AUDIT['piece_oracle']

print(f'gallery   : {PIECE["n_gallery"]} sentences')
print(f'tokeniser : {PIECE["tokenizer"]}')
print(
    f'coverage  : {PIECE["alignment_coverage"]:.4f}'
    f'{"" if PIECE["alignment_coverage"] > 0.99 else "   <-- below 0.99, the piece counts are partly wrong"}'
)
print()
print(f'{"signature":<10}{"Top-1":>10}{"hits":>9}{"bits":>8}{"unique":>9}')
for name, block in PIECE['oracles'].items():
    hits = block['top1'] * block['n']
    print(
        f'{name:<10}{block["top1"]:>10.4f}{hits:>9.0f}{block["information_bits"]:>8.2f}{block["unique_fraction"]:>9.3f}'
    )
print()
print(f'gate    : {PIECE["gate_signature"]} at Top-1 {PIECE["gate_top1"]:.4f} ({PIECE["gate_bits"]:.2f} bits)')
print(f'ceiling : {PIECE["ceiling_signature"]} at Top-1 {PIECE["ceiling_top1"]:.4f}')

### How to read that

`gate` is the floor a **fixed** sub-token count can actually reach — what this repo builds. `ceiling` is what a
design that sized a word's EEG by its own piece count would have handed over, and is reported so the difference
between the two designs is visible rather than argued.

The number to compare either against is this programme's best measured held-out Top-1: **26 of 700, 3.714%** — and
`docs/RESULTS.md` marks that itself stale pending the length-projection re-measurement.

## Part B · Where each trained run sits against that floor

Optional, and only worth running on encoders you care about. This adds `observed_top1` — the run's own held-out
Top-1 — so the oracle returns a verdict rather than `not measured`. It retrains nothing.

A run already audited from the same checkpoint is skipped, so the loop is safe to re-run after a reclaimed runtime
and picks up exactly where it stopped.

In [ ]:
RUNS = colab('runs', '--drive', ZTE_DRIVE, '--headline')['runs']

evaluated = [r for r in RUNS if r.get('evaluated') and (r['checkpoints'] or {}).get('best')]
print(f'{len(evaluated)} evaluated run(s) with a best.pt:\n')
for r in evaluated:
    print(f'  {r["name"]}')

In [ ]:
# Name the runs to audit, or leave empty to take every evaluated one. Each is roughly five minutes,
# and one already audited from the same checkpoint costs seconds.
WANTED: list[str] = []

TARGETS = [r for r in evaluated if not WANTED or r['name'] in WANTED]
print(f'auditing {len(TARGETS)} run(s)')

for run in TARGETS:
    print(f'\n===== {run["name"]} =====')
    ckpt = run['checkpoints']['best']
    out = f'{AUDIT_DIR}/runs/{run["name"]}'
    !uv run zte-rebaseline --ckpt "{ckpt}" --root "{DATA_DIR}" --piece-oracle --out "{out}"

In [ ]:
import pandas as pd

rows = []
for run in TARGETS:
    path = pathlib.Path(f'{AUDIT_DIR}/runs/{run["name"]}/rebaseline.json')
    if not path.is_file():
        print(f'{run["name"]}: no rebaseline.json — the audit did not finish')
        continue
    report = json.loads(path.read_text())
    piece = report.get('piece_oracle') or {}
    floor = report.get('floor_comparison') or {}
    rows.append(
        {
            'run': run['name'],
            'held_out_top1': piece.get('observed_top1'),
            'piece_gate': piece.get('gate_top1'),
            'clears_piece': piece.get('beats_oracles'),
            'length_oracle': floor.get('oracle'),
            'clears_length': floor.get('clears_floor'),
            'rank_pct': floor.get('encoder'),
            'coverage': piece.get('alignment_coverage'),
        }
    )

display(pd.DataFrame(rows))

## Part C · Package and download

One zip, small enough to attach anywhere. It carries the JSON and Markdown this notebook produced and nothing
else — no checkpoints, no embeddings, no raw EEG.

In [ ]:
import shutil

BUNDLE = f'/content/zte_results_audit_{RUN_DATE}'
shutil.make_archive(BUNDLE, 'zip', AUDIT_DIR)

archive = pathlib.Path(f'{BUNDLE}.zip')
print(f'{archive.name}  ·  {archive.stat().st_size / 1024:.0f} KB')
print(f'also kept on Drive at {AUDIT_DIR}')

In [ ]:
from google.colab import files  # type: ignore[import-untyped]

files.download(f'{BUNDLE}.zip')

Unzip it into `res/audits/` in the checkout:

```sh
mkdir -p res/audits && unzip -o ~/Downloads/zte_results_audit_*.zip -d res/audits/
```

`res/` is gitignored, so nothing here is committed — it is working reference, and `docs/` stays the authority.